
<img src="img/viu_logo.png" width="200">

## 01MIAR - Procesamiento de Datos

![logo](img/python_logo.png)

*Ivan Fuertes / Franklin Alvarez*

<div style="background-color:#e7f5ff;border-left:6px solid #1c7ed6;padding:12px 16px;border-radius:6px;margin:6px 0;"><b style="color:#1864ab;font-size:14px;">TEORIA &middot; Como leer este notebook (version mejorada)</b>

Este notebook fue **enriquecido** para ayudarte a aprender escribiendo codigo. Vas a encontrar tres tipos de bloques de colores:

<div style="background-color:#e7f5ff;border-left:6px solid #1c7ed6;padding:6px 10px;border-radius:4px;margin:4px 0;display:inline-block;"><b>AZUL = TEORIA</b></div> -> explicacion en palabras simples antes de cada concepto.

<div style="background-color:#fff9db;border-left:6px solid #f59f00;padding:6px 10px;border-radius:4px;margin:4px 0;display:inline-block;"><b>AMARILLO = TU TURNO</b></div> -> el ejercicio que TU debes resolver. Hay una celda vacia debajo donde escribes el codigo.

<div style="background-color:#ebfbee;border-left:6px solid #2f9e44;padding:6px 10px;border-radius:4px;margin:4px 0;display:inline-block;"><b>VERDE = SOLUCION</b></div> -> codigo de referencia comentado por si te trabas. Destapa solo despues de intentar.

> **Promesa importante**: el codigo y la teoria del profesor original no se han tocado. Solo se han ANADIDO bloques. Asi puedes seguir comparando con la version original cuando quieras.

</div>

<div style="background-color:#e7f5ff;border-left:6px solid #1c7ed6;padding:12px 16px;border-radius:6px;margin:6px 0;"><b style="color:#1864ab;font-size:14px;">TEORIA &middot; Procesamiento de datos: el 80% del trabajo real</b>

En proyectos de Machine Learning e Inteligencia Artificial, **el 70-80% del tiempo se dedica a preparar los datos**, no a entrenar modelos. La calidad del modelo nunca puede superar la calidad de los datos: *garbage in, garbage out*.

#### Que problemas tipicos aparecen en los datos crudos?

| Problema | Que significa | Tipico en |
|---|---|---|
| **Valores faltantes (NaN)** | Celdas vacias o nulas | Encuestas, sensores, bases con campos opcionales |
| **Duplicados** | Filas repetidas | Combinacion de fuentes, errores de captura |
| **Outliers** | Valores extremos atipicos | Errores de medicion, casos raros, fraudes |
| **Formatos inconsistentes** | "Madrid" vs "madrid" vs "MADRID" | Datos cargados manualmente |
| **Tipos incorrectos** | Numeros guardados como string | CSV mal parseado |
| **Variables categoricas en texto** | Modelo necesita numeros | Genero, pais, nivel educativo |
| **Escalas distintas** | Una columna [0,1], otra [0,1_000_000] | Algoritmos de distancia (KNN, K-means, redes) |

#### Las cuatro etapas tipicas de un pipeline de procesamiento

1. **Limpieza** -> quitar/imputar NaN, eliminar duplicados, detectar outliers, corregir formatos.
2. **Transformacion** -> derivar nuevas columnas, separar/concatenar campos, parsear strings.
3. **Encoding** -> convertir variables categoricas a numericas (one-hot, label, ordinal).
4. **Normalizacion / Escalado** -> poner todas las columnas numericas en escalas comparables (min-max, z-score).

Este notebook te lleva por las herramientas de pandas para resolver cada una. Los **TU TURNO** te haran practicar sobre datasets reales.

</div>

In [ ]:
import numpy as np
import pandas as pd

# Duarante esta sesión mostraremos números flotantes con una precisión de 2 decimales
pd.set_option('display.precision', 2)

### Uniendo datasets con 'join' y 'merge'
- merge() == join()
 - 'join' utiliza por defecto los índices para unir
- Utilizando el parámetro 'on'
 - Si las columnas difieren, 'left_on' y 'right_on'
 
https://miro.medium.com/v2/resize:fit:1400/1*GigXPhr4Ue2zbrgIIoB8Lw.png

<div style="background-color:#e7f5ff;border-left:6px solid #1c7ed6;padding:12px 16px;border-radius:6px;margin:6px 0;"><b style="color:#1864ab;font-size:14px;">TEORIA &middot; merge / join: combinar datasets relacionados</b>

Cuando trabajas con datos del mundo real, casi nunca tienes UN unico dataset gigante: tienes **varios datasets relacionados** por una clave en comun (ej: `UserID`, `MovieID`, `ProductCode`).

`pd.merge()` (y `DataFrame.join()`) replican el `JOIN` de SQL.

#### Tipos de union (parametro `how`)

| `how` | Resultado | Equivalente SQL |
|---|---|---|
| `'inner'` | Solo filas con clave en **ambos** | `INNER JOIN` |
| `'left'` | Todas las del izquierdo + las que matchen del derecho | `LEFT JOIN` |
| `'right'` | Todas las del derecho + las que matchen del izquierdo | `RIGHT JOIN` |
| `'outer'` | **Todas** las filas de los dos (rellena NaN donde no haya match) | `FULL OUTER JOIN` |

#### Parametros importantes

```python
pd.merge(df_left, df_right,
         on='UserID',         # columna comun (debe llamarse igual)
         left_on='id_user',   # si tienen distinto nombre
         right_on='user_id',
         how='inner')         # tipo de union
```

> **Cuidado:** si la clave no es unica en alguno de los dos lados, el numero de filas resultante puede **multiplicarse** (es la causa numero uno de bugs en pipelines de datos).

</div>

#### Combinar varios datasets 
- En base a un elemento en común (índice)
- MovieLens 'UserId'

<div style="background-color:#fff9db;border-left:6px solid #f59f00;padding:12px 16px;border-radius:6px;margin:6px 0;"><b style="color:#a05a00;font-size:14px;">TU TURNO 1</b>

Crea dos DataFrames manuales (puedes hacerlos pequenos):

```python
clientes = pd.DataFrame({
    'cliente_id': [1, 2, 3, 4],
    'nombre':     ['Ana', 'Pedro', 'Lucia', 'Mario']
})
pedidos = pd.DataFrame({
    'cliente_id': [1, 1, 2, 5],
    'producto':   ['libro', 'taza', 'mochila', 'reloj'],
    'precio':     [12.5, 8.0, 30.0, 45.0]
})
```

1. Haz un `merge` **inner** entre los dos.
2. Haz un `merge` **left** (que pasa con Lucia y Mario que no tienen pedidos?).
3. Haz un `merge` **outer** (que pasa con el cliente_id 5 que no esta en clientes?).
4. Cuenta cuantas filas tiene cada resultado y explica por que.

<i>Escribe tu codigo en la celda que sigue.</i></div>

In [ ]:
# Escribe aqui tu codigo:
import pandas as pd

clientes = pd.DataFrame({
    'cliente_id': [1, 2, 3, 4],
    'nombre':     ['Ana', 'Pedro', 'Lucia', 'Mario']
})
pedidos = pd.DataFrame({
    'cliente_id': [1, 1, 2, 5],
    'producto':   ['libro', 'taza', 'mochila', 'reloj'],
    'precio':     [12.5, 8.0, 30.0, 45.0]
})



In [ ]:
# === Solucion sugerida ===
# import pandas as pd
#
# # 1) inner: solo coincidencias en AMBOS lados (cliente_id 1 y 2)
# inner = pd.merge(clientes, pedidos, on='cliente_id', how='inner')
# print('INNER:', len(inner), 'filas')
# print(inner)
#
# # 2) left: todos los clientes; Lucia y Mario aparecen con NaN en pedidos
# left = pd.merge(clientes, pedidos, on='cliente_id', how='left')
# print('LEFT:', len(left), 'filas')
# print(left)
#
# # 3) outer: incluye cliente 5 (no esta en clientes) -> nombre NaN
# outer = pd.merge(clientes, pedidos, on='cliente_id', how='outer')
# print('OUTER:', len(outer), 'filas')
# print(outer)
#
# # Conclusion:
# # - inner descarta filas sin match (ojo si tu pipeline depende de ello)
# # - left/right preserva el lado elegido y rellena con NaN
# # - outer es el union total y suele introducir mas NaN


In [ ]:
import zipfile as zp # para descomprimir archivos zip
import urllib.request # para descargar de URL
import os

# descargar MovieLens dataset
url = 'http://files.grouplens.org/datasets/movielens/ml-1m.zip'  
local_zip = os.path.join("res", "ml-1m.zip")
urllib.request.urlretrieve(url, local_zip)

# descomprimiendo archivo zip
with zp.ZipFile(local_zip, 'r') as zipp: 
    print('Extracting all files...') 
    zipp.extractall(os.path.join("res")) # destino
    print('Done!') 

In [ ]:
ruta_users = os.path.join("res", "ml-1m", "users.dat")
ruta_ratings = os.path.join("res", "ml-1m", "ratings.dat")
ruta_movies = os.path.join("res", "ml-1m", "movies.dat")

users_dataset = pd.read_csv(ruta_users, sep='::', index_col=0,
    header=None, names=['UserID','Gender','Age','Occupation','Zip-code'], engine='python', encoding="ISO-8859-1")

ratings_dataset = pd.read_csv(ruta_ratings, sep='::', index_col=0, 
    header=None, names=['UserID','MovieID','Rating','Timestamp'], engine='python', encoding="ISO-8859-1")

movies_dataset = pd.read_csv(ruta_movies, sep='::', index_col=0, 
    header=None, names=['MovieID','Title','Genre'], engine='python', encoding="ISO-8859-1")

In [ ]:
display(users_dataset.head(5))
print(len(users_dataset))

In [ ]:
display(ratings_dataset.sample(5))
print(len(ratings_dataset))

In [ ]:
display(movies_dataset.sample(5))
print(len(movies_dataset))

In [ ]:
# combinando users y ratings, ¿Cómo?
combined_dataset = users_dataset.merge(ratings_dataset, on='UserID', how='inner') # parametro 'on' define la columna pivote
display(combined_dataset.head(5))
print(len(combined_dataset))

In [ ]:
# combinando movies y el resto
all_dataset = combined_dataset.merge(movies_dataset, on='MovieID', how='outer')
display(all_dataset.sample(5))
print(len(all_dataset))

mask = pd.isnull(all_dataset["Rating"])
all_dataset[mask]["MovieID"].unique()

#### Concatenate
https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.concat.html

## Pivot
- Representar los datos en función a varios parámetros, agregando
```python
pivot_table(<lista de valores>, index=<agregador primario>, columns=<agregador secundario>)
```
- https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.pivot_table.html
- https://pandas.pydata.org/pandas-docs/stable/user_guide/reshaping.html

<div style="background-color:#e7f5ff;border-left:6px solid #1c7ed6;padding:12px 16px;border-radius:6px;margin:6px 0;"><b style="color:#1864ab;font-size:14px;">TEORIA &middot; pivot_table: resumir datos en una tabla cruzada</b>

Una **tabla pivote** te permite ver un valor numerico **agregado** por dos dimensiones a la vez (filas y columnas), como una tabla dinamica de Excel.

```python
df.pivot_table(values='Rating',     # columna que se agrega
               index='Gender',       # filas
               columns='Age',        # columnas
               aggfunc='mean')       # como agregar (default: mean)
```

#### Funciones de agregacion comunes (`aggfunc`)

| `aggfunc` | Que calcula |
|---|---|
| `'mean'` | Media (por defecto) |
| `'sum'` | Suma |
| `'count'` | Cuantas observaciones |
| `'min'` / `'max'` | Minimo / maximo |
| `'std'` / `'var'` | Desviacion estandar / varianza |
| `'median'` | Mediana |
| Lista: `['mean', 'count']` | Varias a la vez |
| Funcion lambda | Personalizada |

#### Ejemplo: ratings medios por genero y edad

Te ayuda a responder preguntas como: *los hombres jovenes valoran las peliculas mas alto que las mujeres mayores?*. Es la base del **analisis exploratorio (EDA)** y muchas veces detecta sesgos en los datos antes de modelar.

</div>

<div style="background-color:#fff9db;border-left:6px solid #f59f00;padding:12px 16px;border-radius:6px;margin:6px 0;"><b style="color:#a05a00;font-size:14px;">TU TURNO 2</b>

Sobre el dataset `all_dataset` que ya construyeron arriba (con users + ratings + movies):

1. Calcula la **media de Rating** por `Occupation` (filas) y `Gender` (columnas) usando `pivot_table`.
2. Calcula el **conteo de ratings** por `Occupation` y `Gender` (cambia `aggfunc='count'`).
3. Identifica que combinacion de `Occupation` x `Gender` ha dado el rating medio mas alto (puedes usar `.stack().idxmax()`).

<i>Escribe tu codigo en la celda que sigue.</i></div>

In [ ]:
# Escribe aqui tu codigo (reusa all_dataset definido arriba):



In [ ]:
# === Solucion sugerida ===
# # 1) Rating medio por Occupation x Gender
# tabla_media = all_dataset.pivot_table('Rating',
#                                       index='Occupation',
#                                       columns='Gender',
#                                       aggfunc='mean')
# display(tabla_media)
#
# # 2) Conteo de ratings
# tabla_conteo = all_dataset.pivot_table('Rating',
#                                       index='Occupation',
#                                       columns='Gender',
#                                       aggfunc='count')
# display(tabla_conteo)
#
# # 3) Combinacion con la media mas alta
# # stack convierte la tabla en serie; idxmax devuelve la (Occupation, Gender)
# combinacion_top = tabla_media.stack().idxmax()
# print('Combinacion con rating medio mas alto:', combinacion_top)
# print('Valor:', tabla_media.stack().max())


In [ ]:
display(all_dataset.pivot_table('Rating', index='Gender', columns='Age'))

# Operación equivalente para cada celda
mask = (all_dataset['Gender'] == 'F') * (all_dataset['Age'] == 1)
F1 = all_dataset[mask].Rating.mean()
print("Valor para F-1:", F1)

In [ ]:
# Podemos agregar funciones o lista de funciones con las que operar
# display(all_dataset.pivot_table('Rating', index='Gender', columns='Age', aggfunc='count'))
display(all_dataset.pivot_table('Rating', index='Gender', columns='Age', aggfunc=['count', 'max', 'mean']))

## Agrupaciones
- agg -> funciones estadísticas de agregación
- Series.unique() -> valores únicos
- pd.value_counts -> ocurrencias

<div style="background-color:#e7f5ff;border-left:6px solid #1c7ed6;padding:12px 16px;border-radius:6px;margin:6px 0;"><b style="color:#1864ab;font-size:14px;">TEORIA &middot; Agrupaciones y conteos: explorar la composicion de los datos</b>

Antes de cualquier transformacion, conviene **conocer** los datos. Estas son las herramientas basicas:

| Metodo | Para que sirve |
|---|---|
| `Series.unique()` | Lista de **valores distintos** que aparecen |
| `Series.nunique()` | **Cuantos** valores distintos hay |
| `Series.value_counts()` | Frecuencia de cada valor (ordenado desc) |
| `Series.value_counts(normalize=True)` | Frecuencia **relativa** (porcentaje) |
| `df.groupby(col).agg(...)` | Agrupa por una columna y aplica funciones |
| `df.describe()` | Estadisticos basicos (count, mean, std, min, max, cuartiles) |
| `df.info()` | Tipos de datos y memoria |

#### Patron clasico: groupby + agg

```python
df.groupby('Gender').agg({
    'Rating': ['mean', 'count', 'std'],
    'Age':    'mean'
})
```

Esto te da, **por cada genero**, varias estadisticas a la vez.

#### Detectar problemas con value_counts

```python
df['Pais'].value_counts()
# Espana    1200
# espana       8     <- error de capitalizacion!
# ESPANA       3
# Spain        5     <- idioma diferente
```

Solo con `value_counts` ya detectaste 4 problemas que un `==` ingenuo no ve.

</div>

In [ ]:
valores_unicos = all_dataset[mask].Rating.unique()
print(valores_unicos)

In [ ]:
ocurrencias = all_dataset[mask]["Rating"].value_counts()
ocurrencias.name = "Número de ocurrencias por cada Rating"
print(ocurrencias)

## Manipulación de strings
```python
split(): separar en bloques en función de un carácter
replace(): reemplazar un carácter por otro
index(): encontrar la posición de un carácter
```

<div style="background-color:#e7f5ff;border-left:6px solid #1c7ed6;padding:12px 16px;border-radius:6px;margin:6px 0;"><b style="color:#1864ab;font-size:14px;">TEORIA &middot; Manipulacion de strings en pandas: el accesor <code>.str</code></b>

En pandas, los metodos de string se aplican a **una columna entera** (Series) usando el accesor `.str`. Asi evitas escribir bucles.

```python
df['nombre'].str.upper()           # todo en mayusculas
df['nombre'].str.lower()           # todo en minusculas
df['nombre'].str.strip()           # quita espacios en bordes
df['nombre'].str.replace('a','@')  # reemplaza
df['nombre'].str.contains('Ana')   # bool: contiene?
df['nombre'].str.startswith('A')   # bool: empieza con?
df['nombre'].str.split('|')        # lista de partes
df['nombre'].str.len()             # longitud
df['nombre'].str[0]                # primer caracter de cada fila
df['nombre'].str[:3]               # tres primeros
```

#### Limpieza tipica de strings

| Problema | Solucion |
|---|---|
| Espacios extra: `'  Madrid '` | `.str.strip()` |
| Mayusculas mezcladas | `.str.lower()` o `.str.title()` |
| Tildes / caracteres raros | `.str.normalize('NFKD')` + filtrado |
| Separadores varios `'a;b,c'` | `.str.replace(';',',').str.split(',')` |
| Necesitas extraer un patron | `.str.extract(r'(...)')` con regex |

> **Buena practica:** crea una funcion `limpiar_texto(s)` que aplique strip + lower + replace de tildes, y aplicala con `.apply` a todas las columnas de texto al cargar el CSV. Te ahorra horas de bugs.

</div>

<div style="background-color:#fff9db;border-left:6px solid #f59f00;padding:12px 16px;border-radius:6px;margin:6px 0;"><b style="color:#a05a00;font-size:14px;">TU TURNO 3</b>

Tienes esta serie con nombres de ciudades cargada de forma sucia:

```python
ciudades = pd.Series(['  Madrid', 'madrid ', 'MADRID', 'BARCELONA', 'Barcelona  ', 'valencia', 'Valencia'])
```

Aplicale las transformaciones necesarias para que **todas las versiones de Madrid** queden iguales (lo mismo con las otras). Despues, usa `value_counts` para ver el conteo final.

> Pista: necesitas combinar `.str.strip()` y `.str.lower()` (o `.str.title()`).

<i>Escribe tu codigo en la celda que sigue.</i></div>

In [ ]:
# Escribe aqui tu codigo:
import pandas as pd
ciudades = pd.Series(['  Madrid', 'madrid ', 'MADRID', 'BARCELONA', 'Barcelona  ', 'valencia', 'Valencia'])



In [ ]:
# === Solucion sugerida ===
# import pandas as pd
# ciudades = pd.Series(['  Madrid', 'madrid ', 'MADRID', 'BARCELONA', 'Barcelona  ', 'valencia', 'Valencia'])
#
# # 1) Quitar espacios en los bordes
# # 2) Pasar todo al mismo case (title -> 'Madrid', 'Barcelona', ...)
# ciudades_limpias = ciudades.str.strip().str.lower().str.title()
# print(ciudades_limpias)
# print()
# print(ciudades_limpias.value_counts())
# # Resultado: Madrid 3, Barcelona 2, Valencia 2


### Ejemplo MovieLens: Separar géneros y año en columnas individuales

In [ ]:
# Ejemplo con MovieLens: Genre
## 1: obtener todos los géneros por separado
## 2: crear un dataset de géneros
## 3: por película, marcar género por separado
## 4: Extraer el año de cada peléicula y colocar en columna individual
## 5: Remover el año del título
## 6: unir con dataset genre
display(movies_dataset.head(3))

In [ ]:
all_genres = movies_dataset['Genre'].apply(lambda x : x.split('|'))
print(all_genres)

In [ ]:
genres = {genre for x in all_genres for genre in x}
print("Set con todos los géneros:")
print(genres)
print()

# Alternativa usando unique()
genres = pd.unique(pd.Series(all_genres.sum()))
print("Lista con todos los géneros:")
print(genres)

In [ ]:
# crear tabla con columnas por género
zeros = np.zeros( (len(movies_dataset), len(genres)) )
genres_frame = pd.DataFrame(zeros, columns=genres, index=list(range(1, len(movies_dataset) + 1)))
display(genres_frame.head(3))

In [ ]:
columns_genres = genres_frame.columns # lista de generos (columnas)
print(columns_genres)

# para cada película, marcar género con 1
for i, genre in enumerate(movies_dataset['Genre']):
    inds = columns_genres.get_indexer(genre.split('|')) # retorna los indices correspondientes a los generos de cada pelicula
    genres_frame.iloc[i,inds] = 1 # localiza las columnas del genero correspondiente, marca con 1

display(genres_frame.head(5))

#### Replace e index para extraer el año de la película

In [ ]:
display(movies_dataset.head(2))

In [ ]:
# extraer el año de la columna Title
def split_year(title):
    index = title.index('(')
    return title[index:].replace('(','').replace(')','')
    
# crear nueva columna Year
movies_dataset['Year'] = movies_dataset['Title'].apply(split_year)
display(movies_dataset.head(2))

In [ ]:
# eliminar el año de la columna Title
def remove_year(title):
    index = title.index('(')
    return title[:index-1].rstrip()

movies_dataset['Title'] = movies_dataset['Title'].apply(remove_year)
display(movies_dataset.head(2))

In [ ]:
# unir con dataset original
movies_split_genre = movies_dataset.join(genres_frame)

display(movies_split_genre.head(5))

In [ ]:
# Remover vieja columna de genre
movies_split_genre.drop('Genre', axis=1, inplace=True)
display(movies_split_genre.head(5))

#### Ejemplo con Regular Expressions
Algunos métodos str aceptan expresiones regulares cómo parámetros. Por ejemplo `match()` y `extract()`
- https://docs.python.org/3/library/re.html
- https://regex101.com/
- import re

<div style="background-color:#e7f5ff;border-left:6px solid #1c7ed6;padding:12px 16px;border-radius:6px;margin:6px 0;"><b style="color:#1864ab;font-size:14px;">TEORIA &middot; Expresiones regulares (regex): patrones para texto</b>

Las **regex** son pequenos lenguajes para describir patrones en texto. En pandas se usan con `.str.match`, `.str.contains`, `.str.extract` y `.str.replace`.

#### Atomos basicos

| Patron | Significa |
|---|---|
| `.` | Cualquier caracter (excepto salto de linea) |
| `\d` | Un digito (0-9) |
| `\w` | Letra, digito o `_` |
| `\s` | Espacio en blanco |
| `[abc]` | a, b o c |
| `[^abc]` | Cualquier caracter EXCEPTO a, b, c |
| `[a-z]` | Cualquier minuscula |

#### Cuantificadores

| Patron | Significa |
|---|---|
| `*` | 0 o mas veces |
| `+` | 1 o mas veces |
| `?` | 0 o 1 vez (opcional) |
| `{5}` | exactamente 5 |
| `{3,7}` | entre 3 y 7 |

#### Anclas

| Patron | Significa |
|---|---|
| `^` | Inicio de string |
| `$` | Final de string |

#### Grupos

| Patron | Significa |
|---|---|
| `(...)` | Grupo de captura (lo que extraes) |
| `(?:...)` | Grupo SIN captura |

#### Ejemplos tipicos

```python
'^\d{5}$'          # exactamente 5 digitos -> codigos postales espanoles
'^[\w\.-]+@[\w\.-]+\.\w+$'   # email
'\b\d{4}\b'       # un numero de 4 digitos (anio) entre limites de palabra
'(\d{2})/(\d{2})/(\d{4})'   # fecha dd/mm/yyyy con tres grupos
```

> **Tip:** [regex101.com](https://regex101.com/) te muestra que matchea cada parte de tu regex en tiempo real. Imprescindible al aprender.

</div>

##### Ejemplo con `match`: Eliminar zip-code con formato erróneo 

In [ ]:
# ¿Cómo localizar que 'Zip-code' tiene un formato erróneo?
users_dataset.sample(5)

In [ ]:
# Regular expression: ^\d{5}$
# ^ = start of the string
# \d = decimal string
# {5} = 5 repeticiones de decimales
# $ = end of string

mask_correcto = users_dataset['Zip-code'].str.match('^\\d{5}$') # Formato correcto
mask_incorrecto = mask_correcto == False
print("Existen {} entradas con el formato incorrecto".format(mask_incorrecto.sum()))

display(users_dataset[mask_incorrecto])

In [ ]:
index_incorrecto = users_dataset[mask_incorrecto].index
print(index_incorrecto)
users_dataset_clean = users_dataset.drop(index_incorrecto)
print("Tamaño del dataset antes de la limpieza:", users_dataset.shape)
print("Tamaño del dataset después de la limpieza:", users_dataset_clean.shape)

##### Ejemplo con `extract`: Extraer año

In [ ]:
movies_dataset = pd.read_csv(ruta_movies, sep='::', index_col=0, 
    header=None, names=['MovieID','Title','Genre'], engine='python', encoding="ISO-8859-1")
display(movies_dataset.head(2))

In [ ]:
# ¿Cómo extraer el año con regex en el formato adecuado?
# Regular expression: (\d{4})
# (= busca apertura parentesis
# \d = decimal string
# {4} = 4 repeticiones de decimales
# ) = cierre de parentesis

movies_year = movies_dataset['Title'].str.extract('(\\d{4})')
movies_dataset['Year'] = movies_year
movies_dataset['Title'] = movies_dataset['Title'].apply(remove_year)

display(movies_dataset.head(2))

## Ejercicio de operaciones con colecciones: Filtrar películas de 1975
```python
reduce: aplicar una operación y retornar un valor
map: aplicar  una operación y retornar una secuencia
filter: retorna una secuencia con elementos que cumplen una condición
```

<div style="background-color:#e7f5ff;border-left:6px solid #1c7ed6;padding:12px 16px;border-radius:6px;margin:6px 0;"><b style="color:#1864ab;font-size:14px;">TEORIA &middot; reduce, map y filter: programacion funcional aplicada a datos</b>

Estas tres funciones vienen de la **programacion funcional**. Aunque pandas tiene equivalentes mas idiomaticos, conocerlas te ayuda a leer codigo de otros y a entender mejor `apply`.

| Funcion | Entrada | Salida | Para que |
|---|---|---|---|
| `reduce(f, iter)` | iterable | **un valor** | Acumular (suma, producto, max...) |
| `map(f, iter)` | iterable | iterable | Transformar cada elemento |
| `filter(f, iter)` | iterable + condicion | iterable | Quedarte con los que cumplen |

| Funcional | Pandas equivalente |
|---|---|
| `reduce(lambda x,y: x+y, lista)` | `Series.sum()`, `.any()`, `.all()` |
| `map(lambda x: x*2, lista)` | `Series.apply(lambda x: x*2)` |
| `filter(lambda x: x>0, lista)` | `Series[Series > 0]` |

> En 99% de los casos en pandas, **prefiere las soluciones nativas** (`apply`, mascaras booleanas, agregaciones). Son mas rapidas y mas legibles. `reduce/map/filter` son utiles en Python puro.

</div>


#### Reduce
- Aplicar una operación matemática a cada uno de los elementos de una colección
- Diferente de 'apply()' porque retorna un valor numérico
- Ejemplo: Detección de géneros en años específicos

https://docs.python.org/3/library/functools.html

In [ ]:
from functools import reduce # necesario para reduce

lista = [1, 3, 5, 7, 9]
print(reduce(lambda x,y: x + y, lista))

In [ ]:
movies_1975 = movies_split_genre[ movies_split_genre['Year'].str.contains('1975') ]
display(movies_1975.head(4))

In [ ]:
any_drama = reduce(lambda x,y : bool(x) | bool(y), movies_1975['Animation']) # hay algún drama en 1975
print(any_drama)

all_comedy = reduce(lambda x,y : bool(x) & bool(y),movies_1975['Comedy']) # son todas las películas de 1975 comedias?
print(all_comedy)

In [ ]:
print(movies_1975['Animation'].any()) # Comprueba si hay algún valor que puede cumplir  
print(movies_1975['Comedy'].all()) # Comprueba si todos los valores son True

In [ ]:
# Observar el tipo de dato antes para ver si es posible aplicar las funciones
print(movies_1975.dtypes)
print()
print("Sólo 2 valores únicos para cualquier género:", movies_1975['Comedy'].unique())

#### Map
- Aplicar  una operación (lambda) y retornar una secuencia
- Verifica que géneros aparecen en las películas de 1975
- Elimina las columnas de los géneros que no aparecen

In [ ]:
mapa = map(lambda x : (x, movies_1975[x].any()), movies_1975)
print(mapa) # Objeto no realizado con los resultados de la operación lambda

display(pd.Series(mapa)) # Normalmente se realiza con pd.Series

In [ ]:
mapa = map(lambda x : movies_1975[x].any(), movies_1975)

columnas_validas = movies_1975.columns[pd.Series(mapa)]
movies_1975_clean = movies_1975[columnas_validas]
display(movies_1975_clean.head(4))

#### Filter
- retorna una secuencia con elementos que cumplen una condición
- Ejemplo: obtener las películas de 1975 que contienen 'The' en el título

In [ ]:
filtro = filter(lambda x : 'The' in x, movies_1975_clean['Title'])

for movie in filtro:
    print(movie)
# ¿Están todos los títulos con "The"? si tiene mayúsculas o no...

In [ ]:
filtro = filter(lambda x : 'the' in x, movies_1975['Title'].str.lower())
list(filtro)

## Cambiar tipo de datos

<div style="background-color:#e7f5ff;border-left:6px solid #1c7ed6;padding:12px 16px;border-radius:6px;margin:6px 0;"><b style="color:#1864ab;font-size:14px;">TEORIA &middot; Tipos de datos en pandas: por que importan</b>

Cada columna (`Series`) tiene un **dtype** que determina:

- **Que operaciones puedes hacer** (no puedes sumar strings).
- **Cuanta memoria ocupa** (`int8` ocupa 8x menos que `int64`).
- **Velocidad** (operaciones vectorizadas en `int` son rapidisimas).

#### Tipos comunes

| dtype pandas | Tipo Python | Ejemplo |
|---|---|---|
| `int64` / `int32` / `int8` | int | Edades, contadores |
| `float64` / `float32` | float | Precios, scores, NaN |
| `bool` | bool | Mascaras, flags |
| `object` | str / mixto | Texto generico (lento) |
| `string` (nuevo) | str | Texto explicito (mejor) |
| `category` | categorias | Genero, pais (con pocos valores distintos) |
| `datetime64[ns]` | datetime | Fechas, timestamps |
| `timedelta64[ns]` | timedelta | Duraciones |

#### Convertir tipos

```python
df['edad'] = df['edad'].astype(int)
df['precio'] = df['precio'].astype('float32')
df['fecha'] = pd.to_datetime(df['fecha'])
df['categoria'] = df['categoria'].astype('category')
```

#### Cuando usar `category`?

Cuando una columna tiene **pocos valores distintos repetidos muchas veces** (ej: genero con 'M'/'F'/'Otro' en un millon de filas). Pandas guarda los valores una sola vez y referencia con un entero. **Ahorra muchisima memoria**.

</div>

In [ ]:
movies_1975_bool = pd.DataFrame()
for col in movies_1975_clean:
    if col in genres:
        movies_1975_bool[col] = movies_1975_clean[col].astype(bool)
    else:
        movies_1975_bool[col] = movies_1975_clean[col]

display(movies_1975_bool.head(4))

## Transformación de variables (calidad de datos)
- Valores no definidos
- Valores duplicados

<div style="background-color:#e7f5ff;border-left:6px solid #1c7ed6;padding:12px 16px;border-radius:6px;margin:6px 0;"><b style="color:#1864ab;font-size:14px;">TEORIA &middot; Limpieza de datos: el corazon del preprocesado</b>

Antes de entrenar cualquier modelo necesitas detectar y resolver:

#### 1. Valores faltantes (NaN, None, NaT)

**Como detectarlos:**
```python
df.isnull().sum()              # cuantos NaN por columna
df.isnull().sum().sum()        # total de NaN en el DataFrame
df.isnull().mean() * 100       # % de NaN por columna
df[df['col'].isnull()]         # filas con NaN en 'col'
```

**Como tratarlos:** ver el bloque siguiente sobre **imputacion**.

#### 2. Duplicados

```python
df.duplicated()                # bool por fila
df.duplicated().sum()          # cuantos
df.drop_duplicates()           # quitar duplicados completos
df.drop_duplicates(subset=['email'])   # duplicados solo en 'email'
df.drop_duplicates(keep='first' | 'last' | False)
```

#### 3. Outliers (valores extremos)

**Detectar (regla del rango intercuartil, IQR):**
```python
Q1 = df['col'].quantile(0.25)
Q3 = df['col'].quantile(0.75)
IQR = Q3 - Q1
lim_inf = Q1 - 1.5 * IQR
lim_sup = Q3 + 1.5 * IQR
outliers = df[(df['col'] < lim_inf) | (df['col'] > lim_sup)]
```

**Tratar:** eliminar, capar (clip), o transformar (log).

#### 4. Formatos inconsistentes

`.str.strip().str.lower()`, regex con `.str.replace(r'\s+', ' ', regex=True)`, parseo de fechas con `pd.to_datetime`.

</div>

<div style="background-color:#e7f5ff;border-left:6px solid #1c7ed6;padding:12px 16px;border-radius:6px;margin:6px 0;"><b style="color:#1864ab;font-size:14px;">TEORIA &middot; Imputacion: rellenar valores faltantes</b>

**Imputar** = sustituir un NaN por un valor razonable. La estrategia depende del tipo de variable y del contexto.

| Estrategia | Cuando usar | Codigo pandas |
|---|---|---|
| **Eliminar fila** | Pocos NaN, datos suficientes | `df.dropna()` |
| **Eliminar columna** | Columna con > 50% NaN | `df.dropna(axis=1)` |
| **Media** (mean) | Numerica simetrica, sin outliers | `df['col'].fillna(df['col'].mean())` |
| **Mediana** (median) | Numerica con outliers o sesgada | `df['col'].fillna(df['col'].median())` |
| **Moda** (mode) | Categorica o discreta | `df['col'].fillna(df['col'].mode()[0])` |
| **Valor fijo** | Sabes el valor 'por defecto' | `df['col'].fillna(0)` o `'desconocido'` |
| **Forward fill** (ffill) | Series temporales | `df['col'].ffill()` |
| **Backward fill** (bfill) | Series temporales | `df['col'].bfill()` |
| **Interpolacion** | Series numericas continuas | `df['col'].interpolate()` |
| **Imputacion por grupo** | NaN distintos por categoria | `df.groupby('cat')['col'].transform(lambda x: x.fillna(x.median()))` |
| **Modelo (KNN, MICE)** | Datasets pequenos / criticos | `sklearn.impute.KNNImputer` |

#### Tabla de decision rapida

| Tu variable es... | Empieza por |
|---|---|
| Numerica simetrica | **Media** |
| Numerica sesgada o con outliers | **Mediana** |
| Categorica | **Moda** o `'desconocido'` |
| Serie temporal | **ffill / bfill / interpolate** |
| > 40% de NaN | Considera **eliminar la columna** |

> **Atencion:** la imputacion **se calcula sobre el train**, no sobre todo el dataset (si no, cometes *data leakage*). En sklearn usa `SimpleImputer` dentro de un `Pipeline`.

</div>

In [ ]:
matrix = pd.DataFrame(np.random.randint(10,size=(5,10)).astype(float))
matrix[matrix < 2] = np.nan
display(matrix)

In [ ]:
# isnull() e isna() son equivalentes

na_col = matrix.isnull().sum()
na_col.name = "Nulos por columna"
print(na_col)
print()

na_row = matrix.isna().sum(axis=1)
na_row.name = "Nulos por fila"
print(na_row)

In [ ]:
# Cantidad valores nulos
print(matrix.isnull().sum().sum())

In [ ]:
# numero de no nulos
na_col = matrix.count()
na_col.name = "Valores válidos por columna"
print(na_col)
print()

na_row = matrix.count(axis=1)
na_row.name = "Valores válidos por fila"
print(na_row)

In [ ]:
# Representación de las filas en las que una determinada columna tiene nulos
display(matrix[matrix[6].isnull()])

In [ ]:
# Conteo de valores que aparecen en el dataset
valores = [2, 8]
# Identificación de valores de dominio que se encuentran en un listado
display(matrix[matrix[9].isin(valores)])

In [ ]:
display(matrix)

In [ ]:
## Tratamiento de valores nulos
print("Eliminar filas con NaN")
display(matrix.dropna(axis=0))

print("Eliminar columnas con NaN")
display(matrix.dropna(axis=1))

<div style="background-color:#fff9db;border-left:6px solid #f59f00;padding:12px 16px;border-radius:6px;margin:6px 0;"><b style="color:#a05a00;font-size:14px;">TU TURNO 4</b>

Crea este DataFrame con valores faltantes y trabajalo:

```python
import numpy as np
import pandas as pd
df = pd.DataFrame({
    'edad':    [25, np.nan, 30, 22, np.nan, 41, 28, np.nan, 35, 50],
    'salario': [30000, 45000, np.nan, 28000, 50000, np.nan, 33000, 42000, np.nan, 80000],
    'ciudad':  ['Madrid', 'Barcelona', np.nan, 'Madrid', 'Valencia', 'Madrid', np.nan, 'Barcelona', 'Sevilla', 'Madrid']
})
```

1. Cuantos NaN tiene cada columna? (usa `isnull().sum()`).
2. Imputa `edad` con la **mediana** (es numerica con un valor extremo, 50).
3. Imputa `salario` con la **media**.
4. Imputa `ciudad` con la **moda** (la mas frecuente).
5. Comprueba que ya no hay NaN.

<i>Escribe tu codigo en la celda que sigue.</i></div>

In [ ]:
# Escribe aqui tu codigo:
import numpy as np
import pandas as pd
df = pd.DataFrame({
    'edad':    [25, np.nan, 30, 22, np.nan, 41, 28, np.nan, 35, 50],
    'salario': [30000, 45000, np.nan, 28000, 50000, np.nan, 33000, 42000, np.nan, 80000],
    'ciudad':  ['Madrid', 'Barcelona', np.nan, 'Madrid', 'Valencia', 'Madrid', np.nan, 'Barcelona', 'Sevilla', 'Madrid']
})



In [ ]:
# === Solucion sugerida ===
# import numpy as np
# import pandas as pd
# df = pd.DataFrame({
#     'edad':    [25, np.nan, 30, 22, np.nan, 41, 28, np.nan, 35, 50],
#     'salario': [30000, 45000, np.nan, 28000, 50000, np.nan, 33000, 42000, np.nan, 80000],
#     'ciudad':  ['Madrid', 'Barcelona', np.nan, 'Madrid', 'Valencia', 'Madrid', np.nan, 'Barcelona', 'Sevilla', 'Madrid']
# })
#
# # 1) Inspeccion inicial
# print('NaN por columna antes:')
# print(df.isnull().sum())
#
# # 2) Edad con mediana (robusta a outliers)
# mediana_edad = df['edad'].median()
# df['edad'] = df['edad'].fillna(mediana_edad)
#
# # 3) Salario con media (asumimos que esta bien)
# media_salario = df['salario'].mean()
# df['salario'] = df['salario'].fillna(media_salario)
#
# # 4) Ciudad con la moda (categorica)
# moda_ciudad = df['ciudad'].mode()[0]   # mode() devuelve una Series
# df['ciudad'] = df['ciudad'].fillna(moda_ciudad)
#
# # 5) Verificacion
# print('NaN por columna despues:')
# print(df.isnull().sum())
# print()
# print(df)


In [ ]:
# eliminar filas donde no hay al menos un número de valores no NaN
display(matrix)
display(matrix.dropna(thresh=8, axis=0))

In [ ]:
# sustituir por un valor fijo
display(matrix.fillna(-1))

In [ ]:
# sustituir por valor dinámico (copia)
display(matrix)

display(matrix.bfill())
# display(matrix.ffill())

In [ ]:
# sustituir por valor dinámico (interpolación)
display(matrix)
display(matrix.interpolate())

#### Tratar valores duplicados

<div style="background-color:#e7f5ff;border-left:6px solid #1c7ed6;padding:12px 16px;border-radius:6px;margin:6px 0;"><b style="color:#1864ab;font-size:14px;">TEORIA &middot; Duplicados: detectar y eliminar filas repetidas</b>

Las filas duplicadas son frecuentes cuando combinas datasets, exportas/importas varias veces o hay errores de captura. Pueden **inflar artificialmente** las metricas y confundir al modelo.

| Metodo | Que hace |
|---|---|
| `df.duplicated()` | Devuelve un bool por fila (True = es duplicado) |
| `df.duplicated().sum()` | Cuantos duplicados hay en total |
| `df.duplicated(subset=['col1','col2'])` | Considera duplicado solo si se repiten esas columnas |
| `df.drop_duplicates()` | Elimina duplicados (se queda con la primera por defecto) |
| `df.drop_duplicates(keep='last')` | Se queda con la ultima |
| `df.drop_duplicates(keep=False)` | Elimina **todas** las apariciones de los duplicados |
| `df.drop_duplicates(subset=['email'])` | Quita filas con el mismo email aunque otras columnas difieran |

#### Casos tipicos

```python
# Caso 1: filas idensticamente identicas
df = df.drop_duplicates()

# Caso 2: misma persona registrada varias veces (mismo email pero distinto timestamp)
df_unicos = df.sort_values('timestamp').drop_duplicates(subset=['email'], keep='last')

# Caso 3: detectar duplicados antes de decidir
print(df[df.duplicated(subset=['email'], keep=False)].sort_values('email'))
```

</div>

<div style="background-color:#fff9db;border-left:6px solid #f59f00;padding:12px 16px;border-radius:6px;margin:6px 0;"><b style="color:#a05a00;font-size:14px;">TU TURNO 5</b>

Crea este DataFrame con duplicados sutiles:

```python
df = pd.DataFrame({
    'email':    ['ana@x.com', 'pedro@x.com', 'ana@x.com', 'lucia@x.com', 'pedro@x.com', 'mario@x.com'],
    'compras':  [3, 5, 7, 2, 5, 1],
    'fecha':    ['2024-01-01','2024-01-02','2024-03-15','2024-02-10','2024-01-02','2024-04-01']
})
```

1. Cuantas filas son duplicadas exactas? (todas las columnas iguales).
2. Cuantos clientes UNICOS hay (por email)?
3. Quedate con la fila **mas reciente** de cada email (pista: ordena por `fecha` y usa `drop_duplicates(subset=['email'], keep='last')`).

<i>Escribe tu codigo en la celda que sigue.</i></div>

In [ ]:
# Escribe aqui tu codigo:
import pandas as pd
df = pd.DataFrame({
    'email':    ['ana@x.com', 'pedro@x.com', 'ana@x.com', 'lucia@x.com', 'pedro@x.com', 'mario@x.com'],
    'compras':  [3, 5, 7, 2, 5, 1],
    'fecha':    ['2024-01-01','2024-01-02','2024-03-15','2024-02-10','2024-01-02','2024-04-01']
})



In [ ]:
# === Solucion sugerida ===
# import pandas as pd
# df = pd.DataFrame({
#     'email':    ['ana@x.com', 'pedro@x.com', 'ana@x.com', 'lucia@x.com', 'pedro@x.com', 'mario@x.com'],
#     'compras':  [3, 5, 7, 2, 5, 1],
#     'fecha':    ['2024-01-01','2024-01-02','2024-03-15','2024-02-10','2024-01-02','2024-04-01']
# })
#
# # 1) Duplicados exactos: misma fila completa
# # ('pedro@x.com', 5, '2024-01-02') aparece dos veces -> 1 duplicado
# print('Duplicados exactos:', df.duplicated().sum())
#
# # 2) Clientes unicos por email
# print('Clientes unicos:', df['email'].nunique())
#
# # 3) Fila mas reciente por email
# df['fecha'] = pd.to_datetime(df['fecha'])
# df_unicos = (df.sort_values('fecha')
#                .drop_duplicates(subset=['email'], keep='last'))
# print(df_unicos)


In [ ]:
serie = pd.Series(['a','b','c','a','c','a','g'])
print(serie.duplicated())

In [ ]:
df = all_dataset.copy()
display(df.head(3))

# Eliminación de los duplicados en una columna definida
df2 = df.drop_duplicates(subset="Title", keep='last', inplace=False)
df2.info()

## Valores categóricos

<div style="background-color:#e7f5ff;border-left:6px solid #1c7ed6;padding:12px 16px;border-radius:6px;margin:6px 0;"><b style="color:#1864ab;font-size:14px;">TEORIA &middot; Variables categoricas y encoding: pasar texto a numero</b>

Los modelos de Machine Learning trabajan con **numeros**, no con texto. Si tienes columnas como `genero`, `pais`, `nivel_educativo`, necesitas **codificarlas (encoding)**.

#### Tres familias de encoding

##### 1. Label Encoding (etiqueta numerica)

Asigna un numero a cada categoria.

```python
# pandas con .astype('category').cat.codes
df['genero_lab'] = df['genero'].astype('category').cat.codes
# 'F' -> 0, 'M' -> 1, 'Otro' -> 2

# sklearn
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df['genero_lab'] = le.fit_transform(df['genero'])
```

**Cuando usarlo:** SOLO si los valores tienen un **orden natural** (XS < S < M < L < XL). Si no lo tienen y los pasas a un modelo lineal, este interpretara que `2 > 1 > 0` y te hara conclusiones falsas.

##### 2. One-Hot Encoding (binarizar)

Crea una columna nueva por cada categoria, con 0/1.

```python
# pandas
df_oh = pd.get_dummies(df, columns=['pais'])
# crea columnas: pais_Espana, pais_Francia, pais_Italia ...

# sklearn
from sklearn.preprocessing import OneHotEncoder
ohe = OneHotEncoder(sparse_output=False)
ohe.fit_transform(df[['pais']])
```

**Cuando usarlo:** **CASI SIEMPRE** para variables categoricas sin orden natural. Es la opcion por defecto.

> Cuidado con la **explosion dimensional**: si tu columna tiene 1000 valores distintos, generaras 1000 columnas. Soluciones: agrupar categorias raras en `'Otros'`, o usar embeddings.

##### 3. Ordinal Encoding (orden conocido)

Asignas numeros respetando un **orden definido por ti**.

```python
from sklearn.preprocessing import OrdinalEncoder
oe = OrdinalEncoder(categories=[['XS','S','M','L','XL']])
df['talla_ord'] = oe.fit_transform(df[['talla']])
# XS->0, S->1, M->2, L->3, XL->4
```

**Cuando usarlo:** cuando hay un **orden semantico real** que el modelo deberia aprovechar.

#### Tabla resumen: que encoding usar?

| Tipo de variable | Encoding recomendado |
|---|---|
| Sin orden, pocas categorias (<10) | **One-Hot** |
| Sin orden, muchas categorias | One-Hot + agrupar raras / Target encoding |
| Con orden natural | **Ordinal** (con categorias explicitas) |
| Binaria (`Si`/`No`) | Label (mapeada a 0/1) |
| Para arboles (Random Forest, XGBoost) | Label suele bastar (los arboles no asumen orden) |
| Para modelos lineales y redes | One-Hot u Ordinal segun corresponda |

</div>

<div style="background-color:#e7f5ff;border-left:6px solid #1c7ed6;padding:12px 16px;border-radius:6px;margin:6px 0;"><b style="color:#1864ab;font-size:14px;">TEORIA &middot; Normalizacion y escalado: poner las columnas en la misma escala</b>

Cuando tus columnas numericas tienen escalas muy distintas (edad [0,100] vs salario [0,200_000]), los algoritmos basados en **distancias** (KNN, K-Means, SVM, redes neuronales) **se desequilibran**: el salario domina por completo y la edad pasa a ser irrelevante.

#### Dos tecnicas principales

##### Min-Max Scaling (normalizacion al [0, 1])

Formula: `x_new = (x - min) / (max - min)`

```python
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
df_norm = scaler.fit_transform(df[['edad','salario']])

# Equivalente con pandas:
df['edad_norm'] = (df['edad'] - df['edad'].min()) / (df['edad'].max() - df['edad'].min())
```

**Pros:** todo en [0,1], facil de interpretar.
**Contras:** muy sensible a outliers (un valor extremo aplasta los demas).

##### Z-score / Standard Scaling (estandarizacion)

Formula: `x_new = (x - mean) / std`

Resultado: media 0, desviacion 1. Permite valores fuera de [0,1].

```python
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
df_std = scaler.fit_transform(df[['edad','salario']])

# Equivalente con pandas:
df['edad_z'] = (df['edad'] - df['edad'].mean()) / df['edad'].std()
```

**Pros:** menos sensible a outliers, asume distribucion aproximadamente normal.
**Contras:** los valores no estan acotados.

##### Robust Scaling (alternativa a outliers)

Usa **mediana e IQR** en vez de media y std. Util cuando hay muchos outliers.

```python
from sklearn.preprocessing import RobustScaler
```

#### Tabla de decision: que escalado usar?

| Situacion | Escalado |
|---|---|
| Variables sin outliers, distribucion aproximadamente normal | **StandardScaler** (z-score) |
| Variables acotadas conocidas, redes neuronales (entradas en [0,1]) | **MinMaxScaler** |
| Muchos outliers | **RobustScaler** |
| Solo arboles (Random Forest, XGBoost) | **Ningun escalado** -> los arboles son invariantes a escala |
| Imagenes (pixeles 0-255) | Dividir entre 255 -> [0,1] |

#### Atencion: train vs test

```python
# CORRECTO
scaler.fit(X_train)               # SOLO se ajusta con train
X_train_s = scaler.transform(X_train)
X_test_s  = scaler.transform(X_test)   # con los parametros de TRAIN

# INCORRECTO (data leakage)
scaler.fit(X_total)               # NUNCA mires el test al ajustar
```

</div>

<div style="background-color:#fff9db;border-left:6px solid #f59f00;padding:12px 16px;border-radius:6px;margin:6px 0;"><b style="color:#a05a00;font-size:14px;">TU TURNO 6</b>

Trabaja sobre este DataFrame mixto que combina los problemas de encoding y escalado:

```python
import pandas as pd
df = pd.DataFrame({
    'edad':    [25, 32, 45, 22, 60, 35, 28],
    'salario': [30000, 45000, 80000, 25000, 120000, 55000, 32000],
    'pais':    ['Espana', 'Francia', 'Espana', 'Italia', 'Francia', 'Espana', 'Italia'],
    'talla':   ['M', 'L', 'XL', 'S', 'XL', 'M', 'L']
})
```

1. Aplica **One-Hot Encoding** a `pais` con `pd.get_dummies`. Cuantas columnas nuevas aparecen?
2. Aplica **Ordinal Encoding** a `talla` con el orden `['S','M','L','XL']`. Pista: usa un dict y `.map(...)`.
3. Aplica **Min-Max scaling manual** a `edad` para que quede en `[0, 1]`.
4. Aplica **Z-score manual** a `salario`.
5. Muestra el DataFrame final.

<i>Escribe tu codigo en la celda que sigue.</i></div>

In [ ]:
# Escribe aqui tu codigo:
import pandas as pd
df = pd.DataFrame({
    'edad':    [25, 32, 45, 22, 60, 35, 28],
    'salario': [30000, 45000, 80000, 25000, 120000, 55000, 32000],
    'pais':    ['Espana', 'Francia', 'Espana', 'Italia', 'Francia', 'Espana', 'Italia'],
    'talla':   ['M', 'L', 'XL', 'S', 'XL', 'M', 'L']
})



In [ ]:
# === Solucion sugerida ===
# import pandas as pd
# df = pd.DataFrame({
#     'edad':    [25, 32, 45, 22, 60, 35, 28],
#     'salario': [30000, 45000, 80000, 25000, 120000, 55000, 32000],
#     'pais':    ['Espana', 'Francia', 'Espana', 'Italia', 'Francia', 'Espana', 'Italia'],
#     'talla':   ['M', 'L', 'XL', 'S', 'XL', 'M', 'L']
# })
#
# # 1) One-Hot Encoding sobre 'pais'
# # get_dummies crea una columna binaria por cada valor distinto
# df = pd.get_dummies(df, columns=['pais'])
# # Aparecen 3 columnas: pais_Espana, pais_Francia, pais_Italia
#
# # 2) Ordinal Encoding sobre 'talla' con un orden semantico
# orden_talla = {'S': 0, 'M': 1, 'L': 2, 'XL': 3}
# df['talla_ord'] = df['talla'].map(orden_talla)
#
# # 3) Min-Max scaling manual de 'edad' -> rango [0, 1]
# df['edad_minmax'] = (df['edad'] - df['edad'].min()) / (df['edad'].max() - df['edad'].min())
#
# # 4) Z-score / Standard scaling manual de 'salario'
# df['salario_z'] = (df['salario'] - df['salario'].mean()) / df['salario'].std()
#
# # 5) Resultado final
# print(df)


In [ ]:
print(df.info())

In [ ]:
df["Age"] = pd.Categorical(df["Age"])

df.info()

In [ ]:
df["Rating"] = df["Rating"].astype("category")

df.info()

#### Discretización (valores categóricos)
- Tras Series y DataFrame, objeto para categorías: Categorical
```python
categorias = pd.cut(<valores>, <bins>) 
```

<div style="background-color:#e7f5ff;border-left:6px solid #1c7ed6;padding:12px 16px;border-radius:6px;margin:6px 0;"><b style="color:#1864ab;font-size:14px;">TEORIA &middot; Discretizacion: convertir numerico en categorico (binning)</b>

A veces te interesa **agrupar** valores numericos en bandas (bins) y tratarlos como categorias. Es lo que hace el `pd.cut` que ves abajo.

#### Por que discretizar?

- **Modelos mas robustos**: a veces, en vez de la edad exacta, lo que importa es 'menor', 'joven', 'adulto', 'mayor'.
- **Visualizacion**: histogramas, agrupaciones por rango.
- **Reglas de negocio**: tarifas por rango de ingresos, descuentos por edad.
- **Tratar outliers** sin perderlos.

#### Dos funciones clave

| Funcion | Como divide |
|---|---|
| `pd.cut(x, bins=...)` | Por **rangos de igual ancho** (define los limites) |
| `pd.qcut(x, q=...)` | Por **cuantiles** (cada bin tiene aprox el mismo numero de observaciones) |

#### Ejemplo

```python
edades = [16, 25, 18, 71, 44, 100, 12]

# cut: limites explicitos
bins = [0, 18, 35, 65, 99, np.inf]
etiquetas = ['Menor', 'Joven', 'Adulto', 'Mayor', 'Anciano']
pd.cut(edades, bins, labels=etiquetas)

# qcut: 3 grupos del mismo tamano (terciles)
pd.qcut(edades, q=3, labels=['bajo','medio','alto'])
```

#### Cuando usar cada uno?

| Necesitas... | Usa |
|---|---|
| Limites con sentido de negocio (ej: edad legal, percentiles educativos) | **`cut`** con bins definidos |
| Bins equiprobables (mismo numero de observaciones por bin) | **`qcut`** |
| Distribucion uniforme entre min y max | **`cut`** con `bins=N` (entero) |

</div>

<div style="background-color:#fff9db;border-left:6px solid #f59f00;padding:12px 16px;border-radius:6px;margin:6px 0;"><b style="color:#a05a00;font-size:14px;">TU TURNO 7</b>

Tienes esta serie de salarios:

```python
import pandas as pd
salarios = pd.Series([18000, 22000, 25000, 28000, 32000, 38000, 45000, 55000, 70000, 95000, 150000])
```

1. Usa `pd.cut` con bins explicitos `[0, 25000, 40000, 70000, np.inf]` y etiquetas `['Bajo','Medio','Alto','Muy alto']`. Imprime cuantos hay en cada categoria con `value_counts`.
2. Usa `pd.qcut` con `q=4` y etiquetas `['Q1','Q2','Q3','Q4']` (cuartiles). Comprueba que en cada cuartil hay aprox el mismo numero de personas.
3. Que diferencia ves entre los dos enfoques?

<i>Escribe tu codigo en la celda que sigue.</i></div>

In [ ]:
# Escribe aqui tu codigo:
import numpy as np
import pandas as pd
salarios = pd.Series([18000, 22000, 25000, 28000, 32000, 38000, 45000, 55000, 70000, 95000, 150000])



In [ ]:
# === Solucion sugerida ===
# import numpy as np
# import pandas as pd
# salarios = pd.Series([18000, 22000, 25000, 28000, 32000, 38000, 45000, 55000, 70000, 95000, 150000])
#
# # 1) cut: rangos definidos por nosotros (igual ancho del bin)
# bins = [0, 25000, 40000, 70000, np.inf]
# etiquetas = ['Bajo', 'Medio', 'Alto', 'Muy alto']
# cat_cut = pd.cut(salarios, bins=bins, labels=etiquetas)
# print('cut (rangos por valor):')
# print(cat_cut.value_counts())
# # Resultado: bins desiguales en numero de observaciones
#
# # 2) qcut: por cuantiles (mismo numero por bin, distinto rango)
# cat_qcut = pd.qcut(salarios, q=4, labels=['Q1','Q2','Q3','Q4'])
# print('qcut (cuartiles):')
# print(cat_qcut.value_counts())
# # Resultado: aprox el mismo numero por bin (3, 3, 3, 2 con 11 datos)
#
# # 3) Diferencia:
# # - cut respeta los limites de negocio pero algunos bins pueden quedar vacios
# #   o muy poblados.
# # - qcut equilibra el numero de observaciones, util para crear
# #   variables categoricas con poder predictivo equilibrado.


In [ ]:
# especificar los bloques
bins = [0,18,35,65,99, np.inf] # Límites para definir 5 categorías
etiquetas = ["Menor","Joven","Adulto","Mayor","Anciano"]
edades = [16,25,18,71,44,100,12]

categorias = pd.cut(edades,bins,labels=etiquetas)
display(categorias)
print()
print("Verificar si pertenece a una categoría:")
display(categorias == "Menor")

In [ ]:
# Cuantos por categoría?
categorias.value_counts()

In [ ]:
# Categorías por rango de valores
bins = 3
etiquetas = ["Eliminado", "Repechaje", "Clasificado"]
puntos = [0,6,8,16,25,18,71,44,100]

# Utilizar cut me devuelve rangos del mismo tamaño (max - min)/bin
categorias = pd.cut(puntos, bins, labels=etiquetas) 
print(categorias)
print()
print(categorias.value_counts())

In [ ]:
bins = 3
etiquetas = ["Eliminado", "Repechaje", "Clasificado"]
edades = [1,6,8,16,25,18,71,44,100]

# Utilizar qcut me devuelve distribuciones del mismo tamaño
categorias = pd.qcut(edades,bins,labels=etiquetas) # rangos homogéneos (similar número de valores)
print(categorias)
print()
print(categorias.value_counts())